Montgomery County Maryland Wine Sales Dashboard V2

In [1]:
# Validation cell - run this first
import os
import sys

print("Environment Check:")
print(f"Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")
print(f"Utils folder exists: {os.path.exists('./utils')}")

# Check required files
required_files = ['data/wine_data_fully_classified.pkl', 'data/processed_01_cleaned.pkl', 'data/processed_02_enriched.pkl', 'data/wine_sales_with_reviews_FINAL.pkl']
for file in required_files:
    print(f"{file}: {'EXISTS' if os.path.exists(file) else 'MISSING (will be created)'}")

Environment Check:
Python version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Working directory: C:\Users\LOW14\Jupyter\Montgomery_County_Dashboard\V2 Backup
Utils folder exists: False
data/wine_data_fully_classified.pkl: EXISTS
data/processed_01_cleaned.pkl: EXISTS
data/processed_02_enriched.pkl: EXISTS
data/wine_sales_with_reviews_FINAL.pkl: EXISTS


In [2]:
# Core data processing
import pandas as pd
import numpy as np
import pickle
import os
import sys
import warnings
from difflib import SequenceMatcher

# Visualization 
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Date handling
from datetime import datetime

# Custom utilities
sys.path.append('./utils')
import fuzzy_supplier_matching as fuzzy
import enhanced_data_cleaning_utils as deu
import wine_classification_utils as wcu
import wine_review_matching_utils as wrmu

warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'fuzzy_supplier_matching'

In [3]:
# Data loading configuration
BASE_URL = "https://raw.githubusercontent.com/ac604605/Montgomery_County_Dashboard/main/data/"
LOAD_CONFIG = {
    'low_memory': True,
    'dtype': {'License_ID': str, 'Zip_Code': str}
}

# Load datasets
Distributors_Virginia_Three_Main = pd.read_csv(f"{BASE_URL}Distributors_Virginia_Three_Main.csv", **LOAD_CONFIG)
wine_producers = pd.read_csv(f"{BASE_URL}wine_producers.csv", **LOAD_CONFIG)
Warehouse_and_Retail_Sales = pd.read_csv(f"{BASE_URL}Warehouse_and_Retail_Sales.csv", **LOAD_CONFIG)
Wine_Review_Data = pd.read_csv(f"{BASE_URL}winemag-data-130k-v2.csv", **LOAD_CONFIG)

# Fix supplier data structure
SUPPLIER_COLUMNS = ['License_ID', 'Trade Name', 'Address', 'City', 'State', 'Zip_Code', 'Report_Type']
Suppliers_Fixed = pd.read_csv(
    f"{BASE_URL}Suppliers_Importers_Retailers.csv",
    header=0,
    names=SUPPLIER_COLUMNS,
    usecols=range(7),
    **LOAD_CONFIG
)

After investigating the sales data, there are a few cleaning steps that need to take place. First will be removing all values that are not wine and beer items carried by distributors. Second will be ensuring item codes are numeric for easier processesing. Finally, some idividual values will be changed and anything that isn't wine or beer will be removed. I will also be removing the keg versions of wines and beer, as those would introduce greater scope that I do not wish to manage for a simple portfolio. 

In [4]:
# Save as pickle (recommended for data analysis)
#df_working = Warehouse_and_Retail_Sales.copy()
#df_clean, cleaning_report = deu.run_complete_item_code_standardization(
#    df_working, 
#    item_types_to_keep=['WINE', 'BEER']
#)
#df_clean.to_pickle('data/processed_01_cleaned.pkl')

# To load later:
df_clean = pd.read_pickle('data/processed_01_cleaned.pkl')
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 230053 entries, 0 to 307644
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   YEAR              230053 non-null  int64  
 1   MONTH             230053 non-null  int64  
 2   SUPPLIER          230053 non-null  object 
 3   ITEM CODE         230053 non-null  int64  
 4   ITEM DESCRIPTION  230053 non-null  object 
 5   ITEM TYPE         230053 non-null  object 
 6   RETAIL SALES      230053 non-null  float64
 7   RETAIL TRANSFERS  230053 non-null  float64
 8   WAREHOUSE SALES   230053 non-null  float64
dtypes: float64(3), int64(3), object(3)
memory usage: 17.6+ MB


I will also perform some joins and various cleaning of supporting data tables meant to make brand ownership rights more clear. 

In [5]:
# Save as pickle (recommended for data analysis)
#df_enriched = fuzzy.run_supplier_enrichment(
#    df_clean, 
#    Suppliers_Fixed, 
#    test_mode=False
#)
#df_enriched.to_pickle('data/processed_02_enriched.pkl')

# To load later:
df_enriched = pd.read_pickle('data/processed_02_enriched.pkl')
df_enriched.info()

<class 'pandas.core.frame.DataFrame'>
Index: 230053 entries, 0 to 307644
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   YEAR                   230053 non-null  int64  
 1   MONTH                  230053 non-null  int64  
 2   SUPPLIER               230053 non-null  object 
 3   ITEM CODE              230053 non-null  int64  
 4   ITEM DESCRIPTION       230053 non-null  object 
 5   ITEM TYPE              230053 non-null  object 
 6   RETAIL SALES           230053 non-null  float64
 7   RETAIL TRANSFERS       230053 non-null  float64
 8   WAREHOUSE SALES        230053 non-null  float64
 9   MATCHED_SUPPLIER_NAME  230053 non-null  object 
 10  SUPPLIER_MATCH_SCORE   230053 non-null  float64
 11  SUPPLIER_REPORT_TYPE   230053 non-null  object 
dtypes: float64(4), int64(3), object(5)
memory usage: 22.8+ MB


After reviewing the wine review data, it appears this will serve as an adequate database to gather missing country data for our sales table. For the sake of simplicity, I will combine all columns from this table to matching wines in our sales table. Columns we do not need can be filtered out later. 

#very time intensive process set to markdown

full_results, sales_map, review_map = wrmu.run_wine_review_matching(
    df_clean, Wine_Review_Data, threshold=0.6, test_mode=False
)

In [6]:
#very time intensive process!!!!!!
#full_results, sales_map, review_map = wrmu.run_wine_review_matching( df_clean, Wine_Review_Data, threshold=0.6, test_mode=False )# Save as pickle (recommended for data analysis)
#full_results.to_pickle('wine_sales_with_reviews_FINAL.pkl')

# To load later:
df = pd.read_pickle('data/wine_sales_with_reviews_FINAL.pkl')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 187640 entries, 0 to 307637
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   YEAR                 187640 non-null  int64  
 1   MONTH                187640 non-null  int64  
 2   SUPPLIER             187640 non-null  object 
 3   ITEM CODE            187640 non-null  int64  
 4   ITEM DESCRIPTION     187640 non-null  object 
 5   ITEM TYPE            187640 non-null  object 
 6   RETAIL SALES         187640 non-null  float64
 7   RETAIL TRANSFERS     187640 non-null  float64
 8   WAREHOUSE SALES      187640 non-null  float64
 9   review_title         187640 non-null  object 
 10  review_country       187632 non-null  object 
 11  review_variety       187640 non-null  object 
 12  review_points        187640 non-null  object 
 13  review_price         182737 non-null  object 
 14  review_description   187640 non-null  object 
 15  review_province      1

In [7]:
# Save as pickle (recommended for data analysis)
#df_classified, variety_counts, country_counts = wcu.run_enhanced_wine_classification(df)
#df_classified['wine_color'] = df_classified['final_variety'].apply(wcu.classify_wine_color)
#df_complete = df_classified
#df_complete.to_pickle('wine_data_fully_classified.pkl')

# To load later:
df_complete = pd.read_pickle('data/wine_data_fully_classified.pkl')
df_complete['final_variety'] = df_complete['final_variety'].replace('', pd.NA)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 187640 entries, 0 to 307637
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   YEAR                 187640 non-null  int64  
 1   MONTH                187640 non-null  int64  
 2   SUPPLIER             187640 non-null  object 
 3   ITEM CODE            187640 non-null  int64  
 4   ITEM DESCRIPTION     187640 non-null  object 
 5   ITEM TYPE            187640 non-null  object 
 6   RETAIL SALES         187640 non-null  float64
 7   RETAIL TRANSFERS     187640 non-null  float64
 8   WAREHOUSE SALES      187640 non-null  float64
 9   review_title         187640 non-null  object 
 10  review_country       187632 non-null  object 
 11  review_variety       187640 non-null  object 
 12  review_points        187640 non-null  object 
 13  review_price         182737 non-null  object 
 14  review_description   187640 non-null  object 
 15  review_province      1

In [32]:
import pandas as pd

# Load the match cache pickle
df_match_cache = pd.read_pickle(r"C:\Users\LOW14\Jupyter\Montgomery_County_Dashboard\match_cache.pkl")

wine_name = 'TOKAJI ASZU 5 PUTTONOS'

# Old dataset subset
cols_old = [
    'YEAR', 'MONTH', 'ITEM DESCRIPTION', 'WINE_NAME_EXTRACTED',
    'review_title', 'review_country', 'review_variety',
    'review_points', 'review_price'
]
df_old_subset = df_complete_old[df_complete_old['WINE_NAME_EXTRACTED'] == wine_name][cols_old]

# New match cache: filter list of dicts (or DataFrame)
# If your pickle loaded as a list of dicts:
df_new_subset = pd.DataFrame([d for d in df_match_cache if d['wine_name_extracted'] == wine_name])

# Merge old & new info
df_comparison = df_old_subset.copy()
if not df_new_subset.empty:
    df_comparison['review_match_score'] = df_new_subset.iloc[0]['review_match_score']

df_comparison


TypeError: string indices must be integers, not 'str'

In [35]:
import pandas as pd

wine_name = 'TOKAJI ASZU 5 PUTTONOS'

# Columns from old dataset to keep
cols_old = [
    'YEAR', 'MONTH', 'ITEM DESCRIPTION', 'WINE_NAME_EXTRACTED',
    'review_title', 'review_country', 'review_variety',
    'review_points', 'review_price'
]

# Old dataset subset
df_old_subset = df_complete_old[df_complete_old['WINE_NAME_EXTRACTED'] == wine_name][cols_old].copy()

# New match cache entry
df_new_entry = df_match_cache[wine_name]  # dict for this wine
df_new_subset = pd.DataFrame([df_new_entry])  # convert dict to DataFrame

# Optional: rename new columns to distinguish
df_new_subset = df_new_subset.rename(columns={
    'review_country': 'new_review_country',
    'review_variety': 'new_review_variety',
    'review_points': 'new_review_points',
    'review_price': 'new_review_price',
    'review_match_score': 'new_review_match_score',
    'review_title': 'new_review_title'
})

# Merge old & new info: cross join to add new match info to all old records
df_comparison = df_old_subset.copy()
for col in df_new_subset.columns:
    df_comparison[col] = df_new_subset.iloc[0][col]

df_comparison


,YEAR,MONTH,ITEM DESCRIPTION,WINE_NAME_EXTRACTED,review_title,review_country,review_variety,review_points,review_price,sales_id,wine_name_extracted,new_review_match_score,new_review_title,new_review_country,new_review_variety,new_review_points,new_review_price
858,2020,1,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
24608,2020,3,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
36546,2017,6,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
50117,2017,7,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
63156,2017,8,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
76746,2017,9,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
90117,2017,10,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
103764,2017,11,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
118264,2017,12,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
132677,2018,1,TOKAJI ASZU 5 PUTTONOS - 500ML,TOKAJI ASZU 5 PUTTONOS,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0,19642,TOKAJI ASZU 5 PUTTONOS,0.666667,Royal Tokaji 2003 Aszú 5 Puttonyos (Tokaji),Hungary,Tokaji,89,39.0
